# V7 PIR Pipeline (The 'Recall Engine')
Aggressive Candidate Generation + Time-Aware Cross-Validation targeting Precision@10 > 0.2


In [1]:
import polars as pl
import numpy as np
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
import optuna
import gc
import warnings
import re
warnings.filterwarnings('ignore')

T_PATH = '/kaggle/input/datasets/kinonquc/qkindataset2/transaction_full_2025.parquet'
I_PATH = '/kaggle/input/datasets/kinonquc/qkindataset2/items.parquet'


In [2]:
print("=== LOADING DATA ===")
df_raw = pl.read_parquet(T_PATH).select([
    pl.col('customer_id').cast(pl.Int64),
    pl.col('item_id').cast(pl.Utf8),
    pl.col('updated_date').cast(pl.Datetime).alias('event_ts'),
    pl.col('location').cast(pl.Utf8),
    pl.col('price').cast(pl.Float32),
    pl.col('quantity').cast(pl.Float32)
]).with_columns(pl.col('event_ts').dt.month().alias('month'))

items_df = pl.read_parquet(I_PATH).select([
    'item_id', 'category', 'category_l1', 'category_l2', 'category_l3', 'brand', 'size'
]).with_columns(pl.col('item_id').cast(pl.Utf8))

cat_cols = ['category', 'category_l1', 'category_l2', 'category_l3', 'brand']
for c in cat_cols:
    items_df = items_df.with_columns(pl.col(c).fill_null('Unknown').cast(pl.Categorical).to_physical().cast(pl.Int32).alias(f'{c}_id'))

def standardize_age(val):
    if not isinstance(val, str): return -1.0
    val = val.lower().strip()
    match = re.search(r'(\d+)\s*(m|y|tháng|tuổi)', val)
    if match:
        num = float(match.group(1))
        unit = match.group(2)
        if unit in ['m', 'tháng']: return num / 12.0
        return num
    return -1.0

size_map = {row[0]: standardize_age(row[1]) for row in items_df.select(['item_id', 'size']).iter_rows()}
items_df = items_df.with_columns(pl.col('item_id').replace(size_map, default=-1.0).cast(pl.Float32).alias('item_age_proxy'))


=== LOADING DATA ===


In [3]:
from scipy.sparse import csr_matrix
print("=== V7 RETRIEVER: ENSEMBLE CANDIDATE GENERATION ===")
class V7Retriever:
    def __init__(self, history_df, items_df):
        self.history_df = history_df
        self.items_df = items_df
        self.max_ts = history_df['event_ts'].max()
        
        # Pre-calculate item metadata using global context if available
        self.item_locs = history_df.group_by('item_id').agg(pl.col('location').unique().alias('item_hubs'))
        self.item_prcs = history_df.group_by('item_id').agg(pl.col('price').median().alias('item_p'))
        self._build_indexes()
        self._build_trends()

    def _build_indexes(self):
        print(" Building SVD & Cosine I2I...")
        hist = self.history_df.group_by(['customer_id', 'item_id']).agg(pl.col('quantity').sum().alias('w'))
        last_buy = self.history_df.group_by(['customer_id', 'item_id']).agg(pl.col('event_ts').max().alias('last_ts'))
        hist = hist.join(last_buy, on=['customer_id', 'item_id'])
        hist = hist.with_columns((pl.col('w') * (0.9 ** ((self.max_ts - pl.col('last_ts')).dt.total_days() / 30.0))).alias('w'))
        
        hist = hist.with_columns([pl.col('customer_id').rank('dense').cast(pl.Int64).alias('u_idx')-1, pl.col('item_id').rank('dense').cast(pl.Int32).alias('i_idx')-1])
        self.u_map = hist.select(['customer_id', 'u_idx']).unique()
        self.i_map = hist.select(['item_id', 'i_idx']).unique()
        self.u2idx = dict(zip(self.u_map['customer_id'], self.u_map['u_idx']))
        self.items_list = self.i_map.sort('i_idx')['item_id'].to_list()
        
        self.matrix = csr_matrix((hist['w'].to_numpy(), (hist['u_idx'].to_numpy(), hist['i_idx'].to_numpy())), shape=(self.u_map.height, self.i_map.height), dtype=np.float32)
        
        self.svd = TruncatedSVD(n_components=128, random_state=42)
        self.u_vecs = self.svd.fit_transform(self.matrix)
        self.i_vecs = self.svd.components_.T
        
        norm_m = normalize(self.matrix, norm='l2', axis=0)
        self.i2i_cosine = norm_m.T.dot(norm_m).astype(np.float32)
        self.i2i_cosine.setdiag(0)

    def _build_trends(self):
        vol_30d = self.history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=30)).group_by('item_id').len()
        self.global_trend = vol_30d.sort('len', descending=True).head(150).select('item_id')
        self.local_trend = self.history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=60)).group_by(['location', 'item_id']).len().sort(['location', 'len'], descending=[False, True]).group_by('location').head(80)
        j_df = self.history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=60)).join(self.items_df.select(['item_id', 'category_l1']), on='item_id')
        self.cat_trend = j_df.group_by(['category_l1', 'item_id']).len().sort(['category_l1', 'len'], descending=[False, True]).group_by('category_l1').head(30)
        j_df_b = self.history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=90)).join(self.items_df.select(['item_id', 'brand']), on='item_id').filter(pl.col('brand') != 'Không xác định')
        self.brand_trend = j_df_b.group_by(['brand', 'item_id']).len().sort(['brand', 'len'], descending=[False, True]).group_by('brand').head(10)

    def get_candidates(self, target_users):
        hist_s = self.history_df.filter(pl.col('customer_id').is_in(target_users))
        cands = {}
        cands['rep'] = hist_s.select(['customer_id', 'item_id']).unique()
        cands['global'] = pl.DataFrame({'customer_id': target_users}).join(self.global_trend.with_columns(pl.lit(1).alias('_k')), how='cross').drop('_k')
        user_loc = hist_s.group_by('customer_id').agg(pl.col('location').mode().first().alias('location'))
        cands['local'] = user_loc.join(self.local_trend, on='location').select(['customer_id', 'item_id']).unique()
        user_cats = hist_s.join(self.items_df.select(['item_id', 'category_l1']), on='item_id').group_by(['customer_id', 'category_l1']).len().sort(['customer_id', 'len'], descending=[False, True]).group_by('customer_id').head(3)
        cands['cat'] = user_cats.join(self.cat_trend, on='category_l1').select(['customer_id', 'item_id']).unique()
        user_brands = hist_s.join(self.items_df.select(['item_id', 'brand']), on='item_id').filter(pl.col('brand') != 'Không xác định').group_by(['customer_id', 'brand']).len().sort(['customer_id', 'len'], descending=[False, True]).group_by('customer_id').head(3)
        cands['brand'] = user_brands.join(self.brand_trend, on='brand').select(['customer_id', 'item_id']).unique()
        
        t_idx = [self.u2idx[u] for u in target_users if u in self.u2idx]
        t_u = [u for u in target_users if u in self.u2idx]
        i_arr = np.array(self.items_list)
        chunk = 2000
        c_i2i, c_svd = [], []
        for i in range(0, len(t_idx), chunk):
            idx = t_idx[i:i+chunk]
            u_b = np.array(t_u[i:i+chunk])
            s_s = self.u_vecs[idx].dot(self.i_vecs.T)
            t30 = np.argsort(-s_s, axis=1)[:, :30]
            c_svd.append(pl.DataFrame({'customer_id': pl.Series(np.repeat(u_b, 30), dtype=pl.Int64), 'item_id': i_arr[t30.flatten()]}))
            s_i = self.matrix[idx].dot(self.i2i_cosine).toarray()
            t40 = np.argsort(-s_i, axis=1)[:, :40]
            m = np.take_along_axis(s_i, t40, axis=1) > 0
            c_i2i.append(pl.DataFrame({'customer_id': pl.Series(np.repeat(u_b, 40)[m.flatten()], dtype=pl.Int64), 'item_id': i_arr[t40.flatten()][m.flatten()]}))
        cands['svd'] = pl.concat(c_svd).unique() if c_svd else pl.DataFrame(schema={'customer_id': pl.Int64, 'item_id': pl.Utf8})
        cands['i2i'] = pl.concat(c_i2i).unique() if c_i2i else pl.DataFrame(schema={'customer_id': pl.Int64, 'item_id': pl.Utf8})
        
        all_cands = pl.concat([df for df in cands.values() if df.height > 0]).unique()
        uh = hist_s.group_by('customer_id').agg(pl.col('location').mode().first().alias('loc'))
        up = hist_s.group_by('customer_id').agg(pl.col('price').mean().alias('avg_p'))
        f = (all_cands.join(up, on='customer_id', how='left').join(self.item_prcs, on='item_id', how='left').filter((pl.col('item_p') <= pl.col('avg_p') * 6) | (pl.col('avg_p').is_null())).select(['customer_id', 'item_id']))
        item_loc_flat = self.item_locs.explode('item_hubs').rename({'item_hubs': 'loc'})
        f = (f.join(uh, on='customer_id', how='left').join(item_loc_flat, on=['item_id', 'loc'], how='inner').select(['customer_id', 'item_id']))
        return f


=== V7 RETRIEVER: ENSEMBLE CANDIDATE GENERATION ===


In [4]:
def create_dataset_v7(history_df, truth_df, items_df, sample_users=None, n_negatives=50):
    if sample_users:
        valid_u = history_df['customer_id'].unique().shuffle(seed=42).head(sample_users).to_list()
    else:
        valid_u = history_df['customer_id'].unique().to_list()
    
    hist_s = history_df.filter(pl.col('customer_id').is_in(valid_u))
    retriever = V7Retriever(history_df, items_df)
    filtered_cands = retriever.get_candidates(valid_u)
    
    if truth_df is not None:
        truth = truth_df.filter(pl.col('customer_id').is_in(valid_u)).select(['customer_id', 'item_id']).unique()
        ds = filtered_cands.join(truth.with_columns(pl.lit(1).cast(pl.Int8).alias('target')), on=['customer_id', 'item_id'], how='left').fill_null(0)
        missed = truth.join(filtered_cands, on=['customer_id', 'item_id'], how='anti').with_columns(pl.lit(1).cast(pl.Int8).alias('target'))
        ds = pl.concat([ds, missed]).unique(subset=['customer_id', 'item_id'])
        if n_negatives:
            pos = ds.filter(pl.col('target') == 1)
            neg = ds.filter(pl.col('target') == 0).sample(fraction=1.0, shuffle=True, seed=42).group_by('customer_id').head(n_negatives)
            ds = pl.concat([pos, neg]).sort(['customer_id', 'target'], descending=[False, True])
    else:
        ds = filtered_cands
    
    # Features
    u_prof = hist_s.group_by('customer_id').agg([
        pl.col('item_id').n_unique().alias('u_items'),
        pl.col('quantity').sum().alias('u_qty'),
        (history_df['event_ts'].max() - pl.col('event_ts').min()).dt.total_days().alias('u_tenure'),
        pl.col('price').mean().alias('u_avg_p')
    ])
    i_prof = history_df.group_by('item_id').agg([
        pl.col('customer_id').n_unique().alias('i_users'),
        pl.col('quantity').sum().alias('i_qty')
    ])
    ui_hist = hist_s.group_by(['customer_id', 'item_id']).agg([
        pl.col('quantity').sum().alias('ui_qty'),
        pl.col('event_ts').max().alias('ui_last_dt')
    ])
    max_ts = history_df['event_ts'].max()
    ui_hist = ui_hist.with_columns((max_ts - pl.col('ui_last_dt')).dt.total_days().alias('ui_recency_days'))
    vol_7d = history_df.filter(pl.col('event_ts') >= max_ts - pl.duration(days=7)).group_by('item_id').len().rename({'len': 'v7'})
    vol_28d = history_df.filter(pl.col('event_ts') >= max_ts - pl.duration(days=28)).group_by('item_id').len().rename({'len': 'v28'})
    momentum = vol_7d.join(vol_28d, on='item_id', how='left').with_columns((pl.col('v7') / (pl.col('v28') / 4.0 + 1)).alias('item_momentum'))
    rep_cycles = hist_s.sort(['customer_id', 'item_id', 'event_ts']).with_columns(pl.col('event_ts').diff().dt.total_days().over(['customer_id', 'item_id']).alias('gap'))
    u_cat_rep = rep_cycles.filter(pl.col('gap').is_not_null()).join(items_df.select(['item_id', 'category_l1']), on='item_id').group_by(['customer_id', 'category_l1']).agg(pl.col('gap').median().alias('cat_median_gap'))

    ds = ds.join(u_prof, on='customer_id', how='left')
    ds = ds.join(i_prof, on='item_id', how='left')
    ds = ds.join(ui_hist.select(['customer_id', 'item_id', 'ui_qty', 'ui_recency_days']), on=['customer_id', 'item_id'], how='left')
    ds = ds.join(items_df.select(['item_id', 'item_age_proxy'] + cat_cols + [f'{c}_id' for c in cat_cols]), on='item_id', how='left')
    ds = ds.join(momentum.select(['item_id', 'item_momentum']), on='item_id', how='left')
    ds = ds.join(u_cat_rep, on=['customer_id', 'category_l1'], how='left')
    ds = ds.with_columns((pl.col('ui_recency_days') - pl.col('cat_median_gap')).alias('replenishment_delta'))
    return ds.fill_null(0)


In [5]:
print("=== V7 TIME-AWARE CROSS-VALIDATION ===")
def get_fold_data(train_end_month, val_month):
    print(f"Generating Fold: Train <= {train_end_month}, Val {val_month}")
    h = df_raw.filter(pl.col('month') <= train_end_month)
    t = df_raw.filter(pl.col('month') == val_month)
    return create_dataset_v7(h, t, items_df, sample_users=40000, n_negatives=80)

fold1 = get_fold_data(8, 9)
fold2 = get_fold_data(9, 10)
test_set = create_dataset_v7(df_raw.filter(pl.col('month') <= 10), df_raw.filter(pl.col('month') == 11), items_df, sample_users=20000, n_negatives=None)


=== V7 TIME-AWARE CROSS-VALIDATION ===
Generating Fold: Train <= 8, Val 9
 Building SVD & Cosine I2I...
Generating Fold: Train <= 9, Val 10
 Building SVD & Cosine I2I...
 Building SVD & Cosine I2I...


In [6]:
print("=== OPTUNA CV TUNING ===")
cat_feat_ids = [f'{c}_id' for c in cat_cols]
all_feats = ['u_items', 'u_qty', 'u_tenure', 'u_avg_p', 'i_users', 'i_qty', 'ui_qty', 'ui_recency_days', 'item_momentum', 'replenishment_delta', 'item_age_proxy'] + cat_feat_ids

def prep_lgb(df):
    p = df.to_pandas()
    for c in cat_feat_ids: p[c] = p[c].astype('category')
    return p[all_feats], p['target'], p.groupby('customer_id').size().values

X1, y1, g1 = prep_lgb(fold1)
X2, y2, g2 = prep_lgb(fold2)

def objective(trial):
    param = {
        'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [10], 'verbosity': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 6, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 100),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
    }
    # Average NDCG across folds
    scores = []
    # Train on Fold 1, Validate on Fold 2
    dtrain = lgb.Dataset(X1, y1, group=g1)
    dval = lgb.Dataset(X2, y2, group=g2, reference=dtrain)
    m = lgb.train(param, dtrain, valid_sets=[dval], num_boost_round=300, callbacks=[lgb.early_stopping(30)])
    return m.best_score['valid_0']['ndcg@10']

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)
best_params = study.best_params
print("Best Params (CV):", best_params)

X_final = np.concatenate([X1, X2])
y_final = np.concatenate([y1, y2])
g_final = np.concatenate([g1, g2])
d_final = lgb.Dataset(X_final, y_final, group=g_final)
best_params.update({'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [10]})
lgb_m = lgb.train(best_params, d_final, num_boost_round=800)


=== OPTUNA CV TUNING ===


[I 2026-05-16 05:09:32,303] A new study created in memory with name: no-name-9bd82e00-cd8f-49bc-88c4-6cd6af12bf42


Training until validation scores don't improve for 30 rounds


[I 2026-05-16 05:10:22,187] Trial 0 finished with value: 0.8927600298547691 and parameters: {'learning_rate': 0.06379335236719674, 'num_leaves': 137, 'max_depth': 14, 'min_data_in_leaf': 26, 'colsample_bytree': 0.8811466347081476}. Best is trial 0 with value: 0.8927600298547691.


Early stopping, best iteration is:
[51]	valid_0's ndcg@10: 0.89276
Training until validation scores don't improve for 30 rounds


...

Early stopping, best iteration is:
[107]	valid_0's ndcg@10: 0.892644
Training until validation scores don't improve for 30 rounds


[I 2026-05-16 05:24:11,898] Trial 18 finished with value: 0.890663737328792 and parameters: {'learning_rate': 0.015322147166755479, 'num_leaves': 54, 'max_depth': 14, 'min_data_in_leaf': 51, 'colsample_bytree': 0.8488446560405357}. Best is trial 5 with value: 0.8927889101059954.


Early stopping, best iteration is:
[3]	valid_0's ndcg@10: 0.890664
Training until validation scores don't improve for 30 rounds


[I 2026-05-16 05:25:17,224] Trial 19 finished with value: 0.8934007900088072 and parameters: {'learning_rate': 0.04509285993371658, 'num_leaves': 84, 'max_depth': 6, 'min_data_in_leaf': 36, 'colsample_bytree': 0.6472979740504721}. Best is trial 19 with value: 0.8934007900088072.


Early stopping, best iteration is:
[123]	valid_0's ndcg@10: 0.893401
Best Params (CV): {'learning_rate': 0.04509285993371658, 'num_leaves': 84, 'max_depth': 6, 'min_data_in_leaf': 36, 'colsample_bytree': 0.6472979740504721}


In [7]:
print("=== FINAL EVALUATION ===")
X_ts, y_ts, _ = prep_lgb(test_set)
test_set = test_set.with_columns(pl.Series(name='pred', values=lgb_m.predict(X_ts)))

def evaluate(model_col):
    top10 = test_set.sort(['customer_id', model_col], descending=[False, True]).group_by('customer_id', maintain_order=True).head(10)
    truth_map = df_raw.filter(pl.col('month') == 11).filter(pl.col('customer_id').is_in(top10['customer_id'].unique().to_list())).group_by('customer_id').agg(pl.col('item_id'))
    truth_dict = {row[0]: set(row[1]) for row in truth_map.iter_rows()}
    pred_dict = {row[0]: list(row[1]) for row in top10.group_by('customer_id').agg(pl.col('item_id')).iter_rows()}
    h, m, p = 0, 0.0, 0.0
    for uid, truth in truth_dict.items():
        preds = pred_dict.get(uid, [])
        hits = [pr for pr in preds if pr in truth]
        h += len(hits); p += len(hits)/10.0
        for i, pr in enumerate(preds):
            if pr in truth: m += 1.0/(i+1); break
    n = max(1, len(truth_dict))
    return {'Hits': h, 'Precision@10': p/n, 'MRR': m/n}

print(evaluate('pred'))


=== FINAL EVALUATION ===
{'Hits': 9255, 'Precision@10': 0.1967056323060632, 'MRR': 0.604158359057403}
